# NOUBA — Run on Kaggle / Google Colab (16 GB VRAM)
This notebook pulls your NOUBA repository from GitHub, launches **Qwen2.5-14B-Instruct-AWQ** via **vLLM** on a 16 GB GPU (T4), builds the React frontend (auto-installing Node.js 20 if needed), starts FastAPI on port `7860`, and exposes a public HTTPS URL via `cloudflared`.

In [ ]:
# 1. Clone your GitHub repo (replace with your GitHub URL)
REPO_URL = "https://github.com/zaabola/Hackathon_Kizuna.git"
!rm -rf nouba && git clone $REPO_URL nouba
%cd nouba

In [ ]:
# 2. Install vLLM, backend deps, Node 20 (if missing/old on Kaggle), and build frontend
!pip install -q vllm
!pip install -q -r backend/requirements.txt
!bash -c 'if ! command -v node >/dev/null || [ "$(node -v | cut -d. -f1 | tr -d v)" -lt 18 ]; then curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y -qq nodejs; fi'
!cd frontend && npm install --silent && npm run build

In [ ]:
# 3. Launch Qwen2.5-14B-Instruct-AWQ (tuned for 16 GB T4 VRAM) on port 8000
import subprocess, time, urllib.request

vllm_proc = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-14B-Instruct-AWQ",
    "--quantization", "awq",
    "--dtype", "half",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.90",
    "--enforce-eager",
    "--port", "8000"
], stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)

print("Waiting for vLLM server to load Qwen2.5-14B-Instruct-AWQ...")
for _ in range(120):
    try:
        with urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2) as r:
            if r.status == 200:
                print("vLLM is ready!")
                break
    except Exception:
        time.sleep(3)
else:
    print("Timed out waiting for vLLM. Run `!tail -n 40 vllm.log` to inspect.")

In [ ]:
# 4. Start NOUBA FastAPI (API + built React UI on port 7860) and open a non-blocking public URL
import os, re, subprocess, time

env = os.environ.copy()
env["LLM_BASE_URL"] = "http://localhost:8000/v1"
env["LLM_MODEL"] = "Qwen/Qwen2.5-14B-Instruct-AWQ"
env["PYTHONPATH"] = os.path.abspath("backend")

api_proc = subprocess.Popen([
    "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "7860"
], cwd="backend", env=env, stdout=open("fastapi.log", "w"), stderr=subprocess.STDOUT)

!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
!pkill -f cloudflared || true
tunnel_log = open("cloudflared.log", "w")
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--protocol", "http2", "--url", "http://localhost:7860"], stdout=tunnel_log, stderr=subprocess.STDOUT)

public_url = None
registered = False
for _ in range(30):
    time.sleep(1)
    text = open("cloudflared.log", "r", errors="ignore").read()
    if not public_url:
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
        if match:
            public_url = match.group(0)
    if "Registered tunnel connection" in text:
        registered = True
        break

print("Registered:", registered)
print("NOUBA is live at:", public_url or "Check cloudflared.log")